In [ ]:
%pip install peft==0.4.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
!mkdir cache


In [ ]:
!pip install datasets

In [ ]:
!pip install -U datasets huggingface_hub fsspec

**1. Préparation de l'environnement et importations essentielles**


* peft :indispensable pour appliquer les techniques de PEFT comme LoRA.

* datasets : permet de charger et de manipuler facilement des jeux de données de Hugging Face.

* transformers : la bibliothèque centrale pour travailler avec les modèles de langage pré-entraînés (LLMs) et leurs tokenizers.

* torch : la bibliothèque de calcul tensoriel sous-jacente utilisée par transformers (et souvent implicitement requise).

* accelerate : une extension de Hugging Face pour faciliter l'entraînement distribué et l'optimisation des ressources GPU, souvent utile avec le Trainer.

* os et time : modules Python standards pour les opérations système et la gestion du temps, utilisés pour la gestion des fichiers et les noms de dossiers.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, PeftModel
import os
import torch

**2. Chargement du Modèle de Langage Pré-entraîné et de son Tokenizer**


Chargement du modèle bigscience/bloomz-560m et son tokenizer associé. C'est la base sur laquelle nous allons appliquer LoRA.


* Qu'est-ce qu'un modèle causal (AutoModelForCausalLM)?

Un modèle causal de langage est un type de modèle qui prédit le prochain mot (ou token) dans une séquence, en se basant uniquement sur les mots précédents.  Il ne "voit" pas les mots futurs. C'est le modèle idéal pour la génération de texte, car il construit le texte de manière séquentielle, mot par mot. AutoModelForCausalLM.from_pretrained est une classe générique de Hugging Face qui charge automatiquement le bon modèle causal pour un nom de modèle donné.

* À quoi sert un tokenizer?

Un tokenizer est un composant crucial qui convertit le texte brut (humainement lisible) en une séquence de tokens (généralement des nombres entiers) que le modèle peut comprendre et traiter.  Il gère également le processus inverse (dé-tokenisation). Le tokenizer gère des tâches comme :

La segmentation du texte en mots ou sous-mots (tokens).

L'ajout de tokens spéciaux (début de séquence, fin de séquence, padding).

La conversion de ces tokens en identifiants numériques.
AutoTokenizer.from_pretrained sélectionne le tokenizer approprié pour le modèle spécifié. Nous nous assurons également que pad_token est défini pour garantir un traitement correct des séquences de différentes longueurs.

In [ ]:
# --- 2. Chargement du modèle de langage pré-entraîné et de son tokenizer ---
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

# Charger le tokenizer pour le modèle spécifié
tokenizer = AutoTokenizer.from_pretrained(model_name)
# S'assurer que le token de padding est défini, essentiel pour le batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token # Souvent, le token de fin de séquence est utilisé pour le padding

# Charger le modèle de base (le "foundation model")
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Modèle de base chargé : {model_name}")
print(f"Token de padding du tokenizer : {tokenizer.pad_token}, ID : {tokenizer.pad_token_id}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Modèle de base chargé : bigscience/bloomz-560m
Token de padding du tokenizer : <pad>, ID : 3


**3. Chargement et Prétraitement du Jeu de Données**


Nous allons utiliser le dataset "Abirate/english_quotes" pour affiner notre modèle.

* Comment fonctionne la méthode load_dataset?
La fonction load_dataset de la bibliothèque datasets est une interface simple et puissante pour télécharger et charger des jeux de données à partir du Hugging Face Hub (ou de fichiers locaux). Elle gère automatiquement le téléchargement, la mise en cache et le chargement des données dans une structure Dataset facile à manipuler.

* Que signifie le paramètre split="train[:10%]"?
Le paramètre split permet de sélectionner une portion spécifique du jeu de données.

* "train" : désigne la portion d'entraînement du dataset.

* "[:10%]" : est un "slicing" qui indique que nous voulons prendre les 10 premiers pourcents de cette portion d'entraînement. C'est une stratégie d'échantillonnage pour réduire la taille du dataset et accélérer le processus de fine-tuning à des fins de démonstration ou de test, particulièrement utile dans un environnement comme Colab.

* Prétraitement : pourquoi utilise-t-on la fonction map sur le dataset?
La méthode map est une fonction clé de la bibliothèque datasets. Elle permet d'appliquer une transformation (une fonction) à chaque élément ou à chaque lot d'éléments du dataset de manière optimisée. C'est beaucoup plus efficace que de boucler manuellement sur chaque exemple. Ici, nous l'utilisons pour tokeniser chaque citation.

* Comment fonctionne une fonction lambda?
Une fonction lambda en Python est une petite fonction anonyme (sans nom) qui peut prendre n'importe quel nombre d'arguments, mais ne peut avoir qu'une seule expression. Elle est souvent utilisée pour des opérations simples et rapides où définir une fonction def complète serait trop lourd. Dans notre cas, lambda samples: tokenizer(samples["quote"], truncation=True, max_length=512) définit une fonction qui prend un dictionnaire samples, accède à la clé "quote", et applique le tokenizer sur sa valeur.

* truncation=True : s'assure que les séquences trop longues sont coupées.

* max_length=512 : définit la longueur maximale des séquences tokenisées.

* Échantillonnage : Comment sélectionner un petit nombre d'exemples?
Après avoir échantillonné les 10% initiaux avec split="train[:10%]", nous pouvons vouloir une portion encore plus petite pour une visualisation rapide ou des tests. La méthode select permet de le faire.

* Quelle est la différence entre select et d'autres méthodes d'échantillonnage?
select(range(n)) : Extrait un sous-ensemble du dataset en se basant sur une liste d'indices. C'est très précis et utile pour prendre les n premiers éléments, ou des éléments spécifiques.

* shuffle().select(range(n)) : d'abord, mélange le dataset, puis sélectionne les n premiers éléments du dataset mélangé, ce qui donne un échantillon aléatoire.

* train_test_split() : divise le dataset en portions d'entraînement, de validation et de test de manière aléatoire.

* filter() : sélectionne les exemples qui satisfont une certaine condition.

Pour l'affichage, nous prenons un très petit échantillon (select(range(5))) pour ne pas surcharger la console.

In [ ]:
# --- 3. Chargement du dataset et prétraitement ---
print("\nChargement et prétraitement du dataset...")
# Charger le dataset et prendre les 10% premiers de la portion d'entraînement
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

# Mélanger les données pour s'assurer que l'échantillon de 10% est représentatif
data = data.shuffle(seed=42)

# Tokeniser le dataset
# La fonction map applique le tokenizer à chaque "quote" du dataset.
# `batched=True` signifie que la fonction `lambda` reçoit des lots de `samples` pour un traitement plus rapide.
data = data.map(lambda samples: tokenizer(samples["quote"], truncation=True, max_length=512), batched=True)

# Pour l'entraînement, nous avons besoin des colonnes 'input_ids' et 'attention_mask'.
# Nous définirons les 'labels' comme 'input_ids' pour le LM causal.
# Les autres colonnes de texte brut peuvent être supprimées pour l'entraînement.
data = data.remove_columns(["quote", "author", "tags"])


# Afficher un petit échantillon de données prétraitées pour vérification
train_sample_display = data.select(range(5))
print("\nÉchantillon de données prétraitées (5 premiers exemples) :")
print(train_sample_display)


Chargement et prétraitement du dataset...


README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]


Échantillon de données prétraitées (5 premiers exemples) :
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5
})


**4. Configuration de LoRA et Application au Modèle**


C'est ici que nous définissons les paramètres de LoRA et l'appliquons à notre modèle de base.

* LoraConfig
LoraConfig est une classe de la bibliothèque PEFT qui permet de définir les hyperparamètres de LoRA.

* r : rang des matrices d'adaptation LoRA. C'est un paramètre clé qui contrôle le nombre de paramètres ajoutés. Un r plus élevé permet une plus grande expressivité (meilleures performances) au prix de plus de paramètres à entraîner. Les valeurs typiques sont 8, 16, 32, 64. Nous utiliserons r=8.

* lora_alpha : facteur d'échelle pour les matrices LoRA. Généralement, il est défini à une valeur qui est un multiple de r (par exemple, r ou 2*r). Nous utiliserons lora_alpha=16.

* target_modules : liste de noms de modules (couches) dans le modèle de base où les matrices LoRA seront insérées. Pour les modèles de la famille BLOOM, query_key_value est un module courant qui encapsule les projections de requête, clé et valeur de l'attention.

* lora_dropout : taux de dropout appliqué aux couches LoRA pour éviter le surapprentissage. Nous utiliserons 0.1.

* bias="none" : spécifie si les paramètres de biais doivent être entraînés. "none" signifie qu'ils ne sont pas entraînés.

* task_type="CAUSAL_LM" : informe la bibliothèque PEFT du type de tâche pour lequel le modèle est affiné, ce qui peut influencer la configuration interne.

* get_peft_model : prend votre modèle de base (foundation_model) et votre configuration LoRA (lora_config) et retourne une version du modèle où les couches LoRA ont été ajoutées. La majorité des paramètres du modèle d'origine sont gelés, et seuls les petits paramètres ajoutés par LoRA sont rendus entraînables. print_trainable_parameters() est une méthode utile pour visualiser le nombre de paramètres que vous allez réellement entraîner, ce qui devrait être une fraction infime du total du modèle.

In [ ]:
# --- 4. Configuration de LoRA et application au modèle ---
from peft import LoraConfig, get_peft_model

# Définir la configuration LoRA
lora_config = LoraConfig(
    r=8,  # Rang des matrices de mise à jour (hyperparamètre clé)
    lora_alpha=16, # Facteur d'échelle
    target_modules=["query_key_value"], # Modules cibles (spécifique aux modèles BLOOM)
    lora_dropout=0.1, # Taux de dropout pour les couches LoRA
    bias="none", # Ne pas entraîner les paramètres de biais
    task_type="CAUSAL_LM" # Type de tâche pour la fine-tuning
)

# Appliquer les couches LoRA au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)

print("\nParamètres entraînables après application de LoRA :")
# Afficher le nombre de paramètres entraînables (cela doit être une petite fraction du total)
print(peft_model.print_trainable_parameters())


Paramètres entraînables après application de LoRA :
trainable params: 786,432 || all params: 560,001,024 || trainable%: 0.14043402892063284
None


**5. Préparation de l'Entraînement avec le Trainer de Hugging Face**



Le Trainer est l'outil standard de Hugging Face pour l'entraînement des modèles. Il gère la boucle d'entraînement, l'optimisation, l'évaluation et la journalisation.

* TrainingArguments, utilisée pour définir tous les hyperparamètres et configurations de l'entraînement.

* report_to="none" désactive le reporting vers des services externes comme Weights & Biases pour cet exercice.

* output_dir, le répertoire où les logs et les modèles sauvegardés (si save_strategy est activé) seront stockés.

* auto_find_batch_size=True tente de trouver automatiquement la plus grande taille de lot qui tient en mémoire.

* learning_rate, le taux d'apprentissage. Pour LoRA, il est souvent légèrement plus élevé que pour une fine-tuning complète car moins de paramètres sont mis à jour. Nous utilisons 3e-4.

* num_train_epochs, le nombre de passages complets sur le dataset d'entraînement.

* use_cpu=True force l'utilisation du CPU.

* logging_strategy="epoch" et logging_dir : pour journaliser les métriques d'entraînement après chaque époque.

* save_strategy="no" : pour ne pas sauvegarder de checkpoints intermédiaires (nous sauvegarderons le modèle final explicitement).

* DataCollatorForLanguageModeling : fonction ou une classe qui prend une liste d'échantillons du dataset et les combine en un seul lot (batch) prêt pour le modèle. Pour les modèles de langage causaux (mlm=False pour Masked Language Modeling), il s'assure que les labels (les tokens que le modèle doit prédire) sont correctement décalés d'un cran par rapport aux input_ids.

* Initialisation et Lancement de l'Entraînement
Nous instancions le Trainer avec notre modèle PEFT, les arguments d'entraînement, le dataset et le data collator, puis nous appelons la méthode train().

In [ ]:
# --- 5. Préparation de l'entraînement avec le Trainer de Hugging Face ---
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

# Définir le répertoire de sortie pour les logs et les sauvegardes
output_directory = os.path.join("cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True) # Créer le répertoire s'il n'existe pas

# Définir les arguments d'entraînement
training_args = TrainingArguments(
    report_to="none", # Désactiver le reporting vers des services externes
    output_dir=output_directory,
    auto_find_batch_size=True, # Détecter automatiquement la taille de lot optimale
    learning_rate=3e-4, # Taux d'apprentissage adapté pour LoRA
    num_train_epochs=3, # Nombre d'époques d'entraînement
    use_cpu=False, #
    logging_dir=f"{output_directory}/logs", # Répertoire pour les logs
    logging_strategy="epoch", # Journaliser après chaque époque
    save_strategy="no", # Ne pas sauvegarder de checkpoints intermédiaires
)

# Préparer le Data Collator pour le modèle de langage causal
# mlm=False est crucial pour la modélisation de langage causal
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Initialiser le Trainer
trainer = Trainer(
    model=peft_model, # Notre modèle PEFT (avec les couches LoRA)
    args=training_args,
    train_dataset=data, # Notre dataset tokenisé
    data_collator=data_collator,
)

print("\nDémarrage de l'entraînement...")
trainer.train() # Lancer l'entraînement
print("Entraînement terminé !")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



Démarrage de l'entraînement...


Step,Training Loss


Step,Training Loss


Step,Training Loss


Step,Training Loss


Step,Training Loss
63,2.935200
126,2.772900
189,2.667200


Entraînement terminé !


**6. Sauvegarde et Chargement du Modèle LoRA Affiné**


Après l'entraînement, nous sauvegardons les petites matrices LoRA entraînées. Il est important de noter que nous ne sauvegardons pas le modèle complet, mais seulement les adaptations.

* Sauvegarde du modèle
trainer.model.save_pretrained(peft_model_path) : cette fonction est spécifique à PEFT. Elle sauvegarde uniquement les poids des adaptateurs LoRA (lora_A et lora_B ainsi que la lora_config.json) dans le répertoire spécifié. C'est ce qui rend LoRA si efficace en termes de stockage.

* Chargement du modèle pour l'inférence (PeftModel.from_pretrained)
Pour charger un modèle affiné par LoRA pour l'inférence, le processus est en deux étapes :

> Recharger le modèle de base d'origine (AutoModelForCausalLM.from_pretrained(model_name)).

> Charger les adaptateurs LoRA par-dessus ce modèle de base en utilisant PeftModel.from_pretrained(base_model, lora_adapter_path). La bibliothèque peft va alors fusionner logiquement (ou physiquement si vous utilisez merge_and_unload()) les poids de LoRA avec le modèle de base.

* is_trainable=False :s'assure que le modèle chargé n'est pas configuré pour un entraînement ultérieur.

In [ ]:
# --- 6. Sauvegarde du modèle LoRA affiné ---
import time

# Générer un timestamp pour un nom de dossier unique (facilite la gestion des versions)
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Sauvegarder uniquement les adaptateurs LoRA
trainer.model.save_pretrained(peft_model_path)
print(f"\nModèle LoRA affiné sauvegardé dans : {peft_model_path}")




Modèle LoRA affiné sauvegardé dans : cache/working/peft_lab_outputs/peft_model_1753214635


In [ ]:
# --- 6b. Chargement du modèle LoRA sauvegardé pour l'inférence ---
# Recharger le modèle de base d'abord
base_model_for_inference = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer_for_inference = AutoTokenizer.from_pretrained(model_name)
if tokenizer_for_inference.pad_token is None:
    tokenizer_for_inference.pad_token = tokenizer_for_inference.eos_token

# Charger les adaptateurs LoRA par-dessus le modèle de base
loaded_peft_model = PeftModel.from_pretrained(base_model_for_inference, peft_model_path, is_trainable=False)
print("Modèle LoRA affiné chargé pour l'inférence.")

# (Optionnel) Fusionner les poids LoRA dans le modèle de base pour un déploiement plus simple
# Si vous prévoyez de déployer le modèle sans la bibliothèque PEFT, vous pouvez fusionner les adaptateurs
# loaded_peft_model = loaded_peft_model.merge_and_unload()
# print("Poids LoRA fusionnés dans le modèle de base.")

Modèle LoRA affiné chargé pour l'inférence.


**7. Génération de Texte avec le Modèle Affiné**

Enfin, nous testons notre modèle affiné en lui donnant une invite (prompt) et en générant du texte.

* tokenizer("Two things are infinite: ", return_tensors="pt") : Tokenise notre prompt d'entrée et le convertit en tenseurs PyTorch, prêts pour le modèle.

* loaded_peft_model.generate(...) : la méthode principale pour générer du texte.

* max_new_tokens : le nombre maximal de nouveaux tokens à générer.

* num_return_sequences : Le nombre de séquences différentes à générer.

* do_sample=True : active le mode d'échantillonnage stochastique (opposé à la génération déterministe greedy ou beam search), qui rend le texte plus varié.

* temperature : contrôle la "créativité" du modèle. Une valeur plus faible rend le texte plus prévisible, une valeur plus élevée le rend plus aléatoire.

* top_k : limite les choix du modèle aux k tokens les plus probables à chaque étape de génération.

* eos_token_id : indique au modèle de s'arrêter de générer quand il produit le token de fin de séquence.

* tokenizer.batch_decode(...) : convertit les identifiants numériques générés par le modèle en texte lisible. skip_special_tokens=True ignore les tokens spéciaux comme [PAD], [EOS], etc.

In [13]:
# --- 7. Génération de texte avec le modèle affiné ---
# Préparer l'entrée
inputs = tokenizer_for_inference("Two things are infinite: ", return_tensors="pt")

# Déplacer les entrées vers le GPU si disponible
if torch.cuda.is_available():
    loaded_peft_model.to("cuda")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    print("Modèle et entrées déplacés vers le GPU.")
else:
    print("Utilisation du CPU pour la génération (pas de GPU détecté).")


print("\nGénération de texte avec le modèle affiné...")
outputs = loaded_peft_model.generate(
    **inputs,
    max_new_tokens=50, # Nombre maximum de nouveaux tokens à générer
    num_return_sequences=1, # Générer une seule séquence
    do_sample=True, # Activer l'échantillonnage pour plus de diversité
    temperature=0.7, # Contrôler la "créativité"
    top_k=50, # Considérer les 50 tokens les plus probables
    eos_token_id=tokenizer_for_inference.eos_token_id, # Arrêter la génération au token EOS
)

# Décoder le texte généré et l'afficher
generated_text = tokenizer_for_inference.batch_decode(outputs.cpu(), skip_special_tokens=True)
print("\nTexte Généré :")
print(generated_text[0])

Modèle et entrées déplacés vers le GPU.

Génération de texte avec le modèle affiné...

Texte Généré :
Two things are infinite:  time and space. And the universe is infinite. The universe is infinite and there is no end. And there are no limits in it. And the space is infinite. And there is no limit in it. And there is no limit in the universe


Notre système de fine-tuning LoRA a fonctionné comme prévu.

Le message "Modèle et entrées déplacés vers le GPU." confirme que la configuration GPU a été activée et utilisée pour l'inférence, ce qui est excellent pour la performance.

La phrase générée par le modèle :
"Two things are infinite: time and space. And the universe is infinite. The universe is infinite and there is no end. And there are no limits in it. And the space is infinite. And there is no limit in it. And there is no limit in the universe"

Montre que le modèle a bien compris le début de la citation célèbre ("Two things are infinite:...") et a généré une suite cohérente, bien que légèrement répétitive, dans le même thème. Cela prouve que la fine-tuning a eu un effet et que le modèle est capable de générer du texte.


**Conclusion**

Installer les bibliothèques nécessaires : ok

Charger un modèle de base et son tokenizer : ok

Préparer un dataset : ok

Configurer et appliquer LoRA au modèle : ok

Entraîner le modèle affiné avec le Trainer : ok

Sauvegarder et charger le modèle LoRA : ok

Générer du texte avec le modèle affiné : ok